# DPO-17: MT-Bench — SimPO Epoch Trajectory

MT-Bench on all three SimPO epoch checkpoints. Companion to DPO-17's inner-loop
(length + refusal) and forthcoming AE2 LC. Direct trajectory pairing against
DPO-6's DPO trajectory from DPO-8 (same step counts 3732/7464/11196 → end of
each epoch).

**DPO-6 anchors from DPO-8 (MT-Bench, max_new_tokens=1024):**

| Tag | MT-Bench |
|---|---|
| SFT (zephyr template) | 6.29 |
| DPO ep1 | 6.03 |
| DPO ep2 | 6.67 |
| DPO ep3 | **6.82** |
| Zephyr-7B-β (published full-FT) | 7.34 |

**Prediction for SimPO** (per DPO-17 planning, given length should drop):
- MT-Bench likely slightly lower than DPO ep3 (6.5–6.8) — MT-Bench's judge has
  length bias, so shorter SimPO outputs get a small penalty from the judge.
- The interesting comparison is **SimPO LC vs raw on AE2** (next eval), where
  length-bias is explicitly controlled. MT-Bench gives the absolute-scale check.

**Checkpoints:**

| Run ID | Checkpoint | Epoch |
|---|---|---|
| `dpo17_simpo_ep1` | `simpo-3ep-dpo17/checkpoint-3732`  | 1 |
| `dpo17_simpo_ep2` | `simpo-3ep-dpo17/checkpoint-7464`  | 2 |
| `dpo17_simpo_ep3` | `simpo-3ep-dpo17/checkpoint-11196` | 3 |

**Cost:** ~\$5–10 per checkpoint × 3 = ~\$15–30 (GPT-4 judge, 80 questions × 2 turns each)
**Time:** ~30 min per checkpoint = ~90 min total (~8 min merge + ~12–15 min gen + ~10 min judge)
**Pipeline:** merge LoRA → `gen_model_answer` (GPU, free) → `gen_judgment` (GPT-4) → parse → `results/runs.csv`

Same conventions as `mt_bench_dpo8.ipynb` so artifacts/numbers are directly comparable.

In [ ]:
import sys, os, json, subprocess, shutil, tempfile, csv, statistics
from pathlib import Path

REPO_ROOT  = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT))

BASE_MODEL = "mistralai/Mistral-7B-v0.1"
CKPT_ROOT  = REPO_ROOT / "checkpoints"
SIMPO_ROOT = CKPT_ROOT / "simpo-3ep-dpo17"
RUNS_CSV   = REPO_ROOT / "results" / "runs.csv"

# SimPO hparams from trainer_state + simpo_qlora.yaml
SIMPO_HPARAMS = dict(stage="simpo", beta=2.0, epochs=3, lr=5e-6, lora_r=128, simpo_gamma=1.0)

# (checkpoint_path, tag, fastchat model_id, run_id, hparams, notes)
#
# IMPORTANT — model_id MUST contain "zephyr" so FastChat's ZephyrAdapter matches
# and uses the <|user|>/<|assistant|> conversation template. Otherwise it falls
# back to BaseModelAdapter's "one_shot" Vicuna-style template, which is NOT what
# any of these checkpoints were trained on (DPO-5's 5.75 SFT score was this bug).
CHECKPOINTS = [
    (SIMPO_ROOT / "checkpoint-3732",
     "simpo_ep1", "zephyr-simpo-ep1", "dpo17_simpo_ep1",
     SIMPO_HPARAMS, "DPO-17 MT-Bench, SimPO epoch 1"),
    (SIMPO_ROOT / "checkpoint-7464",
     "simpo_ep2", "zephyr-simpo-ep2", "dpo17_simpo_ep2",
     SIMPO_HPARAMS, "DPO-17 MT-Bench, SimPO epoch 2"),
    (SIMPO_ROOT / "checkpoint-11196",
     "simpo_ep3", "zephyr-simpo-ep3", "dpo17_simpo_ep3",
     SIMPO_HPARAMS, "DPO-17 MT-Bench, SimPO epoch 3"),
]

for ckpt, tag, *_ in CHECKPOINTS:
    status = "OK" if ckpt.exists() else "MISSING"
    print(f"  [{status}] {tag}: {ckpt}")

print(f"\nruns.csv: {RUNS_CSV}")

In [ ]:
# Set OPENAI_API_KEY in your shell BEFORE launching Jupyter.
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY not set.\n"
    "PowerShell: $env:OPENAI_API_KEY = 'sk-...'"
)
print(f"OPENAI_API_KEY set ({len(os.environ['OPENAI_API_KEY'])} chars) ✓")

import fastchat
JUDGE_DIR = Path(fastchat.__file__).parent / "llm_judge"
assert JUDGE_DIR.is_dir(), f"llm_judge dir not found: {JUDGE_DIR}"
print(f"llm_judge dir: {JUDGE_DIR}")

# Sanity-check the MT-Bench data files (DPO-5/DPO-8 should have fetched these).
for rel in ["data/mt_bench/question.jsonl",
            "data/mt_bench/reference_answer/gpt-4.jsonl",
            "data/judge_prompts.jsonl"]:
    p = JUDGE_DIR / rel
    print(f"  [{'OK' if p.exists() else 'MISSING'}] {rel}")

## Helpers — same shape as `mt_bench_dpo8.ipynb`

`run_one_checkpoint(...)` does: merge LoRA → gen_model_answer → gen_judgment → parse → append to `runs.csv` → delete merged temp dir. Idempotent on re-run (skips gen/judge if the per-checkpoint output files already exist).

In [ ]:
CSV_FIELDNAMES = [
    "run_id", "checkpoint", "tag", "stage",
    "beta", "epochs", "lr", "lora_r", "simpo_gamma",
    "max_new_tokens",
    "avg_gen_length", "p90_gen_length",
    "harmful_refusal_rate", "over_refusal_rate", "pref_acc",
    "mt_bench", "alpacaeval2_lc", "notes",
]


def stream_run(cmd, cwd=None, auto_confirm=False):
    """Subprocess.Popen with line-buffered passthrough. auto_confirm sends Enter to stdin."""
    proc = subprocess.Popen(
        cmd,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        cwd=str(cwd) if cwd else None,
        env=os.environ,
    )
    if auto_confirm:
        proc.stdin.write("\n")
        proc.stdin.flush()
        proc.stdin.close()
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Failed (exit {proc.returncode}): {' '.join(str(c) for c in cmd)}")


def parse_score(model_id, judge_dir):
    jf = Path(judge_dir) / "data" / "mt_bench" / "model_judgment" / "gpt-4_single.jsonl"
    if not jf.exists():
        raise FileNotFoundError(f"Judgment file not found: {jf}")
    scores = []
    with open(jf) as f:
        for line in f:
            if not line.strip():
                continue
            e = json.loads(line)
            if e.get("model") != model_id:
                continue
            s = e.get("score", -1)
            if isinstance(s, (int, float)) and s != -1:
                scores.append(float(s))
    if not scores:
        raise ValueError(f"No scores found for {model_id!r} in {jf}")
    avg = round(statistics.mean(scores), 2)
    print(f"  {model_id}: {avg:.2f}  ({len(scores)} turns scored)")
    return avg, len(scores)


def append_csv(row: dict):
    write_header = not RUNS_CSV.exists() or RUNS_CSV.stat().st_size == 0
    if RUNS_CSV.exists() and RUNS_CSV.stat().st_size > 0:
        with open(RUNS_CSV, "rb") as f:
            f.seek(-1, 2)
            last_byte = f.read(1)
        if last_byte not in (b"\n", b"\r"):
            with open(RUNS_CSV, "ab") as f:
                f.write(b"\n")
    with open(RUNS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDNAMES, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    print(f"  Appended to {RUNS_CSV}")


def merge_lora(base_model_id: str, lora_path: str, output_dir: Path):
    """Merge a LoRA adapter into the base model and save the full model to disk.

    Runs the merge on GPU in bf16 — CPU-side merging needs ~28 GB RAM and swap-thrashes
    on Windows machines with less, looking like a freeze at "Merging and unloading...".
    The merged model is released from VRAM as soon as it's saved, so FastChat's
    subsequent gen_model_answer load has a clean slate.
    """
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    print(f"  Loading base {base_model_id} (bfloat16, GPU)...")
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id, torch_dtype=torch.bfloat16, device_map="auto"
    )
    print(f"  Attaching LoRA from {lora_path}...")
    model = PeftModel.from_pretrained(base, lora_path)
    print("  Merging and unloading...")
    model = model.merge_and_unload()
    print(f"  Saving merged weights → {output_dir} ...")
    model.save_pretrained(str(output_dir))
    AutoTokenizer.from_pretrained(lora_path).save_pretrained(str(output_dir))
    del model, base
    torch.cuda.empty_cache()
    print(f"  Merged ✓  →  {output_dir}")


def _gen_done(model_id: str) -> bool:
    af = JUDGE_DIR / "data" / "mt_bench" / "model_answer" / f"{model_id}.jsonl"
    if not af.exists():
        return False
    n = sum(1 for line in af.read_text().splitlines() if line.strip())
    return n >= 80  # MT-Bench has 80 questions


def _judge_done(model_id: str) -> bool:
    jf = JUDGE_DIR / "data" / "mt_bench" / "model_judgment" / "gpt-4_single.jsonl"
    if not jf.exists():
        return False
    n = 0
    with open(jf) as f:
        for line in f:
            if not line.strip():
                continue
            e = json.loads(line)
            if e.get("model") == model_id and isinstance(e.get("score", -1), (int, float)) and e["score"] != -1:
                n += 1
    return n >= 160  # 80 questions × 2 turns


def _assert_zephyr_template(model_id: str):
    """Guard against the template-routing bug: model_id must hit ZephyrAdapter."""
    from fastchat.model.model_adapter import get_model_adapter, get_conversation_template
    adapter = get_model_adapter(model_id)
    conv = get_conversation_template(model_id)
    if "zephyr" not in conv.name:
        raise RuntimeError(
            f"model_id={model_id!r} routes to {type(adapter).__name__} "
            f"(template={conv.name!r}). Rename it to contain 'zephyr' so "
            f"ZephyrAdapter matches and the model sees the format it was trained on."
        )
    print(f"  ✓ {model_id} → {type(adapter).__name__} ({conv.name!r})")


def run_one_checkpoint(ckpt_path: Path, tag: str, model_id: str, run_id: str,
                       hparams: dict, notes: str = ""):
    """Full pipeline for one checkpoint. Returns the MT-Bench score."""
    print(f"\n{'='*64}\n  {tag}  ({ckpt_path.name})\n{'='*64}")
    _assert_zephyr_template(model_id)

    # ── Step 1: merge LoRA (skip if generation already done) ────────────────
    merged_dir = None
    if _gen_done(model_id):
        print(f"\n[skip merge] {model_id} answers already exist")
    else:
        merged_dir = Path(tempfile.mkdtemp(prefix=f"mt_bench_{tag}_"))
        print(f"\n[1/3] Merging LoRA → {merged_dir}")
        merge_lora(BASE_MODEL, str(ckpt_path), merged_dir)

    try:
        # ── Step 2: generate model answers ──────────────────────────────────
        if _gen_done(model_id):
            print(f"[skip gen] {model_id}.jsonl already has ≥80 answers")
        else:
            print(f"\n[2/3] gen_model_answer: {model_id}")
            stream_run(
                [sys.executable, "-m", "fastchat.llm_judge.gen_model_answer",
                 "--model-path", str(merged_dir),
                 "--model-id",   model_id,
                 "--bench-name", "mt_bench",
                 "--num-gpus-per-model", "1"],
                cwd=JUDGE_DIR,
            )

        # ── Step 3: judge with GPT-4 ────────────────────────────────────────
        if _judge_done(model_id):
            print(f"[skip judge] {model_id} already has 160 judged turns")
        else:
            print(f"\n[3/3] gen_judgment: {model_id}  (~$5–10)")
            stream_run(
                [sys.executable, "-m", "fastchat.llm_judge.gen_judgment",
                 "--model-list",  model_id,
                 "--judge-model", "gpt-4",
                 "--bench-name",  "mt_bench",
                 "--mode",        "single"],
                cwd=JUDGE_DIR,
                auto_confirm=True,
            )
    finally:
        if merged_dir and merged_dir.exists():
            print(f"\n  Cleaning up {merged_dir}...")
            shutil.rmtree(merged_dir, ignore_errors=True)

    # ── Parse + record ──────────────────────────────────────────────────────
    score, n_turns = parse_score(model_id, JUDGE_DIR)
    append_csv({
        "run_id":     run_id,
        "checkpoint": str(ckpt_path),
        "tag":        tag,
        **hparams,
        "mt_bench":   score,
        "notes":      notes,
    })
    print(f"\n  ▶ {tag} MT-Bench: {score:.2f}  ({n_turns} turns)")
    return score


print("Helpers defined ✓")

## Precheck

Validates every component before any GPU work or API spend (same shape as mt_bench_dpo8.ipynb's precheck — verifies disk space since the merge writes a ~14 GB temp file).

In [ ]:
import time, shutil as _shutil

_errors = []

def _check(name, fn):
    try:
        result = fn()
        msg = f" — {result}" if result else ""
        print(f"  ✓ {name}{msg}")
    except Exception as e:
        print(f"  ✗ {name}: {type(e).__name__}: {e}")
        _errors.append(name)


print("=" * 72)
print(" Precheck — environment · checkpoints · API · CSV write")
print("=" * 72)


def _check_openai():
    from openai import OpenAI
    OpenAI().models.list()
    return "API key valid"
_check("OPENAI_API_KEY pings OpenAI", _check_openai)


def _check_fastchat():
    import fastchat as _fc
    d = Path(_fc.__file__).parent / "llm_judge"
    missing = [r for r in ["data/mt_bench/question.jsonl",
                           "data/mt_bench/reference_answer/gpt-4.jsonl",
                           "data/judge_prompts.jsonl"] if not (d / r).exists()]
    if missing:
        raise FileNotFoundError(f"missing: {missing}")
    return f"v{_fc.__version__}"
_check("fastchat install + MT-Bench data files", _check_fastchat)


def _check_gpu():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available")
    free, total = torch.cuda.mem_get_info()
    free_gb, total_gb = free / 1e9, total / 1e9
    if free_gb < 12:
        raise RuntimeError(f"only {free_gb:.1f} GB free of {total_gb:.1f} GB")
    return f"{torch.cuda.get_device_name(0)}, {free_gb:.1f}/{total_gb:.1f} GB free"
_check("GPU available + ≥12 GB free VRAM", _check_gpu)


def _check_disk():
    tmp = Path(tempfile.gettempdir())
    free_gb = _shutil.disk_usage(tmp).free / 1e9
    if free_gb < 30:
        raise RuntimeError(f"only {free_gb:.1f} GB free at {tmp}")
    return f"{free_gb:.1f} GB free at {tmp}"
_check("Temp disk ≥30 GB free (merged models ~14 GB each)", _check_disk)


def _check_checkpoints():
    for ckpt, *_ in CHECKPOINTS:
        if not ckpt.exists():
            raise FileNotFoundError(ckpt)
        if not (ckpt / "adapter_config.json").exists():
            raise FileNotFoundError(f"{ckpt}/adapter_config.json")
    return f"{len(CHECKPOINTS)} checkpoints OK"
_check("All checkpoint paths + adapter_config.json exist", _check_checkpoints)


def _check_chat_templates():
    from transformers import AutoTokenizer
    for ckpt, tag, *_ in CHECKPOINTS:
        ct = AutoTokenizer.from_pretrained(str(ckpt)).chat_template or ""
        if "<|user|>" not in ct or "<|assistant|>" not in ct:
            raise RuntimeError(f"{tag}: tokenizer chat_template missing Zephyr markers")
    return f"{len(CHECKPOINTS)} tokenizers carry Zephyr template"
_check("Tokenizers have Zephyr chat_template baked in", _check_chat_templates)


def _check_routing():
    from fastchat.model.model_adapter import get_conversation_template
    for _, _, model_id, *_ in CHECKPOINTS:
        conv = get_conversation_template(model_id)
        if "zephyr" not in conv.name:
            raise RuntimeError(f"{model_id!r} routes to {conv.name!r}, not zephyr")
    return f"{len(CHECKPOINTS)} model_ids → ZephyrAdapter"
_check("All model_ids route to ZephyrAdapter", _check_routing)


def _check_csv_write():
    sentinel_id = f"__precheck_{int(time.time())}__"
    backup = RUNS_CSV.read_bytes() if RUNS_CSV.exists() else None
    try:
        append_csv({
            "run_id": sentinel_id, "checkpoint": "precheck", "tag": "precheck",
            "stage": "simpo", "beta": 2.0, "epochs": 3, "lr": 5e-6, "lora_r": 128,
            "simpo_gamma": 1.0, "mt_bench": 0.0,
            "notes": "precheck sentinel — should be rolled back",
        })
        with open(RUNS_CSV, newline="") as f:
            reader = csv.DictReader(f)
            cols = reader.fieldnames
            rows = [r for r in reader if r["run_id"] == sentinel_id]
        if len(rows) != 1:
            raise RuntimeError(f"sentinel not found after append (got {len(rows)} rows)")
        if rows[0]["mt_bench"] != "0.0" or rows[0]["tag"] != "precheck":
            raise RuntimeError(f"column misalignment: {rows[0]}")
        if cols != CSV_FIELDNAMES:
            raise RuntimeError(f"header drift — file has {cols}, code expects {CSV_FIELDNAMES}")
        return f"{len(cols)} columns aligned, sentinel rolled back"
    finally:
        if backup is not None:
            RUNS_CSV.write_bytes(backup)
        else:
            RUNS_CSV.unlink(missing_ok=True)
_check("runs.csv append + alignment + rollback", _check_csv_write)


print("=" * 72)
if _errors:
    print(f"  ✗ {len(_errors)} precheck(s) failed: {_errors}")
    print("    Fix the above before running sections 1–3.")
    raise SystemExit(f"Precheck failed: {_errors}")
else:
    print("  ✓ All prechecks passed — safe to launch MT-Bench")
print("=" * 72)

## 1. MT-Bench — SimPO epoch 1 (`checkpoint-3732`)

Monitor: `python scripts/check_mt_bench.py zephyr-simpo-ep1`

In [ ]:
_ckpt, _tag, _mid, _rid, _hp, _notes = CHECKPOINTS[0]
simpo_ep1_score = run_one_checkpoint(_ckpt, _tag, _mid, _rid, _hp, _notes)

## 2. MT-Bench — SimPO epoch 2 (`checkpoint-7464`)

Monitor: `python scripts/check_mt_bench.py zephyr-simpo-ep2`

In [ ]:
_ckpt, _tag, _mid, _rid, _hp, _notes = CHECKPOINTS[1]
simpo_ep2_score = run_one_checkpoint(_ckpt, _tag, _mid, _rid, _hp, _notes)

## 3. MT-Bench — SimPO epoch 3 (`checkpoint-11196`)

Headline checkpoint. Monitor: `python scripts/check_mt_bench.py zephyr-simpo-ep3`

In [ ]:
_ckpt, _tag, _mid, _rid, _hp, _notes = CHECKPOINTS[2]
simpo_ep3_score = run_one_checkpoint(_ckpt, _tag, _mid, _rid, _hp, _notes)

## 4. Results — SimPO vs DPO MT-Bench trajectory

Joins the three SimPO MT-Bench scores against DPO-6's MT-Bench numbers from DPO-8.
Direct apples-to-apples: same SFT init, same dataset, same step counts, only the
loss function differs.

In [ ]:
# DPO-6 MT-Bench from DPO-8 (results/dpo8_mt_bench_review.md)
MT_DPO = {"ep1": 6.03, "ep2": 6.67, "ep3": 6.82}
MT_SFT = 6.29
MT_ZEPHYR_PUB = 7.34

print("=" * 80)
print(" MT-Bench: DPO trajectory vs SimPO trajectory (DPO-17)")
print("=" * 80)
print(f"{'Checkpoint':<28} {'DPO (DPO-8)':>14} {'SimPO (DPO-17)':>17} {'Δ':>10}")
print("-" * 80)
print(f"{'SFT (zephyr template)':<28} {MT_SFT:>14.2f} {'—':>17} {'—':>10}")
for ep, dpo_score, simpo_score in [
    ("ep1", MT_DPO["ep1"], simpo_ep1_score),
    ("ep2", MT_DPO["ep2"], simpo_ep2_score),
    ("ep3", MT_DPO["ep3"], simpo_ep3_score),
]:
    delta = simpo_score - dpo_score
    print(f"{'  epoch ' + ep[-1]:<28} {dpo_score:>14.2f} {simpo_score:>17.2f} {delta:>+10.2f}")
print(f"{'Zephyr-7B-β (full-FT)':<28} {MT_ZEPHYR_PUB:>14.2f} {'—':>17} {'—':>10}")
print("=" * 80)

best_simpo = max(simpo_ep1_score, simpo_ep2_score, simpo_ep3_score)
best_dpo   = MT_DPO["ep3"]

print(f"\nBest SimPO: {best_simpo:.2f}  vs best DPO (ep3): {best_dpo:.2f}  (Δ {best_simpo - best_dpo:+.2f})")
print(f"QLoRA-vs-full-FT gap (best SimPO vs Zephyr-β): {MT_ZEPHYR_PUB - best_simpo:.2f}")

print("\nNext: AE2 LC eval (notebooks/ae2_dpo17.ipynb) — the headline length-bias")
print("check. SimPO is expected to show much smaller LC−Raw gap than DPO's −2.68 pp")
print("(since length-normalized loss removes the verbosity that inflated DPO's raw).")